# Scryfall Tags: Question Answering  
__Objective:__ Fine tune an LLM to answer questions in the following structure:  
- Q: How would you functionally tag the Magic the Gathering card <card_name>?  
- A: [<tag_1>, <tag_2>, ..., <tag_n>]

## Packages and Data

In [1]:
# packages

## modeling
from transformers import pipeline
from huggingface_hub import notebook_login
from huggingface_hub import Repository
from huggingface_hub import get_full_repo_name

## connect project directory
import sys
from pathlib import Path
dir = str(Path(Path.cwd()).parents[0])
if dir not in sys.path:
    sys.path.append(dir)

## load from project directory
from src.data_gathering.scryfall_qa_dataset import ScryfallQADataset
from src.fine_tuning.training import FineTuneLLM

In [2]:
# login to the hugging face
with open('../huggingface_token.txt', 'r') as f:
    token = f.read()

notebook_login()

In [3]:
# params
from src.config import BUILD_DATASET, TASK, MODEL
from src.config import MAX_INPUT_LENGTH, MAX_TARGET_LENGTH, BATCH_SIZE
from src.config import LEARNING_RATE, GRAD_ACCUMULATION_STEPS, NUM_EPOCHS
from src.config import OUTPUT_DIR

In [4]:
# get dataset
sf = ScryfallQADataset()

## build dataset as needed
if BUILD_DATASET:
    sf.build_dataset(
        task = TASK,
        tag_path = '../reports/scryfall_tags.json',
        train_size_pct = 0.8,
        truncate_dataset = 11,
        test_size_n = 1
    )

## load dataset
sf.load_hf_dataset(
    train_path = '../data/scryfall_summarization_train.json',
    val_path = '../data/scryfall_summarization_val.json',
    test_path = '../data/scryfall_summarization_test.json'
)

Scryfall Tag Question Answering Dataset Built
	Train Records = 8
	Validation Records = 2
	Test Records = 1
	Records saved to...
		../data/scryfall_summarization_train.json
		../data/scryfall_summarization_val.json
		../data/scryfall_summarization_test.json
	NOTE: This method does not create the huggingface dataset object. Run load_dataset() for that.


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Scryfall Tag Question Answering Dataset Loaded
	Train Records = 8
	Val Records = 2
	Test Records = 1


## Modeling

### With Trainer API  
source = https://huggingface.co/learn/llm-course/en/chapter7/5

In [5]:
# fine tune the model

## instantiate the model
finetune = FineTuneLLM(
    model_name = MODEL
)

## prepare data
finetune.prepare_data(
    dataset = sf.dataset,
    max_input_length = MAX_INPUT_LENGTH,
    max_target_length = MAX_TARGET_LENGTH,
    batch_size = BATCH_SIZE
)

## train the model
finetune.train(
    learning_rate = LEARNING_RATE,
    accelerator_mixed_precision = 'fp16',
    accelerator_force_cpu = True,
    accelerator_gradient_steps = GRAD_ACCUMULATION_STEPS,
    num_train_epochs = NUM_EPOCHS
)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

  0%|          | 0/400 [00:00<?, ?it/s]

Epoch 0: {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}
Epoch 1: {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}
Epoch 2: {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}
Epoch 3: {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}
Epoch 4: {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}
Epoch 5: {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}
Epoch 6: {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}
Epoch 7: {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}
Epoch 8: {'rouge1': np.float64(0

In [10]:
print(min(finetune.tokenized_datasets["train"][0]["labels"]))

1


In [7]:
# # upload the model to the huggingface hub

# ## define the repo locally
# ## NOTE: Be sure to create OUTPUT_DIR in the hub manually first
# repo_name = get_full_repo_name(OUTPUT_DIR)
# repo = Repository(OUTPUT_DIR, clone_from = repo_name)

# ## save to hub
# finetune.save_to_huggingface_hub(
#     output_dir = OUTPUT_DIR,
#     repo = repo,
#     commit_message = f'Fine-tuned {MODEL} on scryfall tags.'
# )

## Use Fine-Tuned Model

In [8]:
# load model from the hub
from transformers import pipeline
repo_name = get_full_repo_name(OUTPUT_DIR)
# repo = Repository(OUTPUT_DIR, clone_from = repo_name)
summarizer = pipeline('summarization', model = repo_name)

Device set to use cuda:0


In [9]:
def print_summary(idx):
    card = sf.test['test'][idx]['document']
    actual_tags = sf.test['test'][idx]['summary']
    pred_tags = summarizer(sf.test['test'][idx]['document'])[0]['summary_text']

    print(f'Card = {card}\nActual Tags = {actual_tags}\nPredicted Tags = {pred_tags}')

for i in range(5):
    print(f"\n{'-' * 25}")
    print_summary(i)
    print(f"{'-' * 25}\n")


-------------------------
Card = 
        Nissa, Worldsoul Speaker
        Type Line = Legendary Creature — Elf Druid

        Rules Text = Landfall — Whenever a land you control enters, you get {E}{E} (two energy counters).
You may pay eight {E} rather than pay the mana cost for permanent spells you cast.
 
        Power = 3
Toughness = 3

        
        Color Identity = ['G']

        Rarity = rare
        
Actual Tags = cost ignorer, counter fuel-energy, energy generator, free-cast-another, landfall, triggered ability
Predicted Tags = <extra_id_0>,
-------------------------


-------------------------


IndexError: Invalid key: 1 is out of bounds for size 1

## Citations

@inproceedings{sanh2019distilbert,
  title={DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter},
  author={Sanh, Victor and Debut, Lysandre and Chaumond, Julien and Wolf, Thomas},
  booktitle={NeurIPS EMC^2 Workshop},
  year={2019}
}